# ED Pathway Trainer - Kaggle v2e (pred dump + Brief v5 baked in)
Robust generator + GRU baseline. Includes calibration, gate-tau sweep, stratified CMs, preds dump, and Brief v5 renderer.

In [ ]:
import os, math, json, random
from dataclasses import dataclass, field
from typing import Dict, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
SEED=1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
@dataclass
class GenConfig:
    target_pts_per_day_baseline:int=100
    target_pts_per_day_stress:int=160
    frac_ems:float=0.35
    chest_pain_share_mean:float=0.45
    chest_pain_share_sd:float=0.1
    night_service_mult:Tuple[float,float]=(1.25,1.4)
    weekend_service_mult:Tuple[float,float]=(1.15,1.25)
    night_resource_delta:Dict[str,int]=field(default_factory=lambda:{'lab_analyzers':-1,'xray_rooms':-1,'ct_scanners':0})
    ct_scanners:int=1; xray_rooms:int=2; lab_analyzers:int=2
    svc_ct_shape:float=2.0; svc_ct_scale:float=12.0
    svc_xr_shape:float=2.0; svc_xr_scale:float=6.0
    svc_lab_shape:float=2.0; svc_lab_scale:float=18.0
    stale_burst_every_hours:Tuple[int,int]=(6,12)
    stale_burst_duration_min:Tuple[int,int]=(45,90)
    missing_ts_prob:float=0.05
    duplicate_order_prob:float=0.015
    cancel_prob:float=0.05
    code_flip_prob:float=0.015
    negative_control_frac:float=0.03
    start_datetime:str='2025-08-12 00:00:00'
    include_ecg_fast_as_actions:bool=True

In [ ]:
def sample_daily_arrival_curve(target_total:int):
    base={'morning':np.array([3,3,4,5,6,7,8,9,10,10,9,8,7,7,6,6,6,7,8,6,5,4,3,3],float),
          'evening':np.array([3,3,3,3,3,4,5,6,8,10,12,14,14,14,12,10,9,8,7,6,5,4,3,3],float),
          'flat':np.array([4,4,4,5,5,6,6,7,8,9,9,9,9,9,9,8,8,7,7,6,6,5,5,4],float)}
    arr=base[random.choice(list(base.keys()))].copy()
    arr*=np.random.lognormal(mean=math.log(1.0),sigma=0.35)
    for h in random.sample(range(24),k=random.randint(1,3)): arr[h]*=random.uniform(1.4,1.8)
    arr=arr*(target_total/(arr.sum() or 1.0))
    return np.round(arr).astype(int)

def vary_case_mix(m=0.45,sd=0.1):
    chest=np.clip(np.random.normal(m,sd),0.25,0.65)
    trauma=np.clip(np.random.normal(0.12,0.03),0.05,0.2)
    neuro=np.clip(np.random.normal(0.12,0.04),0.05,0.25)
    other=max(0.0,1.0-(chest+trauma+neuro))
    return {'chest_pain':chest,'polytrauma':trauma,'neuro_deficit':neuro,'other':other}

def daytype(ts):
    is_wk=ts.weekday()>=5; is_n=ts.hour<7 or ts.hour>=19
    if is_wk and is_n: return 'weekend_night'
    if is_wk: return 'weekend_day'
    if is_n: return 'weekday_night'
    return 'weekday_day'

In [ ]:
class ResourceSystem:
    def __init__(self,cfg):
        self.base={'ct':cfg.ct_scanners,'xr':cfg.xray_rooms,'lab':cfg.lab_analyzers}
        self.av={'ct':[0.0]*max(1,cfg.ct_scanners),'xr':[0.0]*max(1,cfg.xray_rooms),'lab':[0.0]*max(1,cfg.lab_analyzers)}
        self.params={'ct':(cfg.svc_ct_shape,cfg.svc_ct_scale),'xr':(cfg.svc_xr_shape,cfg.svc_xr_scale),'lab':(cfg.svc_lab_shape,cfg.svc_lab_scale)}
        self.cfg=cfg
    def _mult(self,d):
        m=1.0
        if 'night' in d: m*=random.uniform(*self.cfg.night_service_mult)
        if 'weekend' in d: m*=random.uniform(*self.cfg.weekend_service_mult)
        return m
    def _active(self,tool,d):
        base=self.base[tool]; delta=0
        if 'night' in d:
            delta=self.cfg.night_resource_delta.get({'ct':'ct_scanners','xr':'xray_rooms','lab':'lab_analyzers'}[tool],0)
        return max(1,base+int(delta))
    def submit(self,tool,arr_min,d):
        k,theta0=self.params[tool]; theta=theta0*self._mult(d)
        dur=float(np.random.gamma(k,theta))
        srv=self.av[tool]; n=self._active(tool,d)
        idx=min(range(n),key=lambda i:srv[i])
        start=max(arr_min,srv[idx]); done=start+dur; srv[idx]=done; return done

In [ ]:
def simulate_one_day(cfg,stress=False,day_index=0):
    target=cfg.target_pts_per_day_stress if stress else cfg.target_pts_per_day_baseline
    arr_h=sample_daily_arrival_curve(target); mix=vary_case_mix(cfg.chest_pain_share_mean,cfg.chest_pain_share_sd)
    st=pd.Timestamp(cfg.start_datetime)+pd.Timedelta(days=day_index)
    next_burst=st+pd.Timedelta(hours=random.randint(*cfg.stale_burst_every_hours)); burst_until=st
    rs=ResourceSystem(cfg); rows=[]
    for h in range(24):
        for _ in range(arr_h[h]):
            m=int(np.random.randint(0,60)); ts=st+pd.Timedelta(hours=h,minutes=m); dtag=daytype(ts)
            r=random.random()
            syn='chest_pain' if r<mix['chest_pain'] else ('polytrauma' if r<mix['chest_pain']+mix['polytrauma'] else ('neuro_deficit' if r<mix['chest_pain']+mix['polytrauma']+mix['neuro_deficit'] else 'other'))
            t0=(ts-st).total_seconds()/60.0
            if syn=='chest_pain':
                lab_s=t0+np.random.normal(15,6); lab_d=rs.submit('lab',lab_s,dtag)
                xr_s=t0+np.random.normal(25,8); xr_d=rs.submit('xr',xr_s,dtag)
                if random.random()<0.15:
                    ct_s=t0+np.random.normal(35,10); ct_d=rs.submit('ct',ct_s,dtag)
                else:
                    ct_s=None; ct_d=None
            elif syn=='polytrauma':
                lab_s=t0+np.random.normal(10,5); lab_d=rs.submit('lab',lab_s,dtag)
                xr_s=t0+np.random.normal(15,6); xr_d=rs.submit('xr',xr_s,dtag)
                ct_s=t0+np.random.normal(20,8); ct_d=rs.submit('ct',ct_s,dtag)
            elif syn=='neuro_deficit':
                lab_s=t0+np.random.normal(12,5); lab_d=rs.submit('lab',lab_s,dtag)
                ct_s=t0+np.random.normal(18,7); ct_d=rs.submit('ct',ct_s,dtag)
                xr_s=None; xr_d=None
            else:
                lab_s=t0+np.random.normal(20,8); lab_d=rs.submit('lab',lab_s,dtag)
                xr_s=t0+np.random.normal(30,10); xr_d=rs.submit('xr',xr_s,dtag)
                ct_s=None; ct_d=None
            consult=0.0
            if syn=='chest_pain':
                if random.random()<0.35: consult=max(0,np.random.normal(30,12))
            elif syn=='neuro_deficit':
                consult=max(0,np.random.normal(15,10))
            if ts>=next_burst and ts>burst_until:
                burst_until=ts+pd.Timedelta(minutes=random.randint(45,90))
                next_burst=ts+pd.Timedelta(hours=random.randint(6,12))
            cap_stale=(ts<=burst_until)
            miss_ts=(random.random()<0.05); cancel=(random.random()<0.05); code_flip=(random.random()<0.015); neg=(random.random()<0.03)
            rows.append({'day_index':day_index,'timestamp':ts.isoformat(),'minute_of_day':int((ts-st).total_seconds()//60),'daytype':dtag,'syndrome':syn,'consult_delay_min':consult,'lab_submit_min':None if miss_ts else lab_s,'lab_done_min':None if miss_ts else lab_d,'xr_submit_min':None if miss_ts or xr_s is None else xr_s,'xr_done_min':None if miss_ts or xr_d is None else xr_d,'ct_submit_min':None if miss_ts or ct_s is None else ct_s,'ct_done_min':None if miss_ts or ct_d is None else ct_d,'cap_stale':cap_stale,'duplicate_order':(random.random()<0.015),'cancel':cancel,'code_flip':code_flip,'negative_control':neg,'ems':(random.random()<0.35)})
    return pd.DataFrame(rows)

In [ ]:
def simulate_days(cfg,n_days=5,stress_ratio=0.35):
    import pandas as pd
    frames=[]
    for d in range(n_days): frames.append(simulate_one_day(cfg,stress=(random.random()<stress_ratio),day_index=d))
    df=pd.concat(frames,ignore_index=True); df.sort_values(['day_index','minute_of_day'],inplace=True); df.reset_index(drop=True,inplace=True); print('Simulated rows:',len(df)); return df
cfg=GenConfig(); df_raw=simulate_days(cfg,5,0.35); df_raw.head()

In [ ]:
ACTIONS=['NO_OP','ORDER_ECG','PERFORM_FAST','ORDER_LABS','ORDER_XR','ORDER_CT','REQUEST_CONSULT','REQUEST_BED']
act2id={a:i for i,a in enumerate(ACTIONS)}

def derive_label(row):
    if row['negative_control']: return act2id['NO_OP']
    t=row['minute_of_day']
    if cfg.include_ecg_fast_as_actions:
        if row['syndrome']=='chest_pain' and random.random()<0.6: return act2id['ORDER_ECG']
        if row['syndrome']=='polytrauma' and random.random()<0.6: return act2id['PERFORM_FAST']
    if row['lab_submit_min'] is not None and abs(row['lab_submit_min']-t)<8: return act2id['ORDER_LABS']
    if row.get('xr_submit_min') is not None and row['xr_submit_min'] is not None and abs(row['xr_submit_min']-t)<8: return act2id['ORDER_XR']
    if row.get('ct_submit_min') is not None and row['ct_submit_min'] is not None and abs(row['ct_submit_min']-t)<8: return act2id['ORDER_CT']
    if row['consult_delay_min']>25 and random.random()<0.5: return act2id['REQUEST_CONSULT']
    if (row.get('ct_done_min') or row.get('xr_done_min') or row.get('lab_done_min')) and (not row['cap_stale']) and random.random()<0.1: return act2id['REQUEST_BED']
    return act2id['NO_OP']

df=df_raw.copy(); df['y']=df.apply(derive_label,axis=1)
feature_cols=['minute_of_day','cap_stale','ems','consult_delay_min']
for syn in ['chest_pain','polytrauma','neuro_deficit','other']:
    df[f'syn_{syn}']=(df['syndrome']==syn).astype(int); feature_cols.append(f'syn_{syn}')
X=df[feature_cols].fillna(0.0).astype(float).values; y=df['y'].values.astype(int)
print('Features:',feature_cols)
print('Class counts:',{ACTIONS[k]:int(v) for k,v in zip(*np.unique(y,return_counts=True))})

In [ ]:
val_mask=(df['day_index']==df['day_index'].max())
train_mask=~val_mask
X_train,y_train=X[train_mask],y[train_mask]
X_val,y_val=X[val_mask],y[val_mask]
print('Train/Val sizes:',X_train.shape,X_val.shape)

In [ ]:
class GRUHead(nn.Module):
    def __init__(self,input_dim,hidden=64,num_classes=8,p_drop=0.3):
        super().__init__(); self.gru=nn.GRU(input_dim,hidden,num_layers=1,batch_first=True); self.dropout=nn.Dropout(p_drop); self.fc=nn.Linear(hidden,num_classes)
    def forward(self,x):
        out,_=self.gru(x); out=out[:,-1,:]; out=self.dropout(out); return self.fc(out)
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,alpha=None,reduction='mean'):
        super().__init__(); self.gamma=gamma; self.alpha=alpha; self.reduction=reduction
    def forward(self,logits,target):
        logp=F.log_softmax(logits,dim=-1); p=logp.exp(); pt=p.gather(1,target.view(-1,1)).squeeze(1); logpt=logp.gather(1,target.view(-1,1)).squeeze(1)
        if self.alpha is not None:
            at=self.alpha.gather(0,target); loss=-at*((1-pt)**self.gamma)*logpt
        else: loss=-((1-pt)**self.gamma)*logpt
        return loss.mean()

def label_smoothed_ce(logits,target,smoothing=0.05):
    n=logits.size(-1); logp=F.log_softmax(logits,dim=-1)
    with torch.no_grad():
        true=torch.zeros_like(logp).fill_(smoothing/(n-1)); true.scatter_(1,target.view(-1,1),1.0-smoothing)
    return torch.mean(torch.sum(-true*logp,dim=-1))

def entropy_reg(probs,strength=0.001):
    ent=-torch.sum(probs*torch.log(probs+1e-8),dim=-1); return strength*torch.mean(ent)

In [ ]:
class TabDataset(Dataset):
    def __init__(self,X,y): self.X=torch.tensor(X,dtype=torch.float32); self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,idx): return self.X[idx].unsqueeze(0), self.y[idx]

train_dl=DataLoader(TabDataset(X_train,y_train),batch_size=256,shuffle=True)
val_dl=DataLoader(TabDataset(X_val,y_val),batch_size=512,shuffle=False)

model=GRUHead(input_dim=X_train.shape[1],hidden=64,num_classes=len(ACTIONS),p_drop=0.3)
opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-3)
use_focal=True
alpha=torch.ones(len(ACTIONS)); alpha[0]=0.25
criterion_focal=FocalLoss(gamma=2.0,alpha=alpha)
num_epochs=5
best_val=1e9
for ep in range(num_epochs):
    model.train(); total=0.0
    for xb,yb in train_dl:
        logits=model(xb); loss=(criterion_focal(logits,yb) if use_focal else label_smoothed_ce(logits,yb,0.05))+entropy_reg(F.softmax(logits,dim=-1),0.0005)
        opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()*len(yb)
    model.eval(); vloss=0.0
    with torch.no_grad():
        for xb,yb in val_dl:
            vloss+=F.cross_entropy(model(xb),yb).item()*len(yb)
    print(f'Epoch {ep+1}: train={total/len(X_train):.4f} val={vloss/len(X_val):.4f}')
    if vloss<best_val: best_val=vloss; torch.save(model.state_dict(),'ga_model.pt')
import torch
model.load_state_dict(torch.load('ga_model.pt',map_location='cpu')); model.eval()

In [ ]:
class TempScale(nn.Module):
    def __init__(self): super().__init__(); self.logT=nn.Parameter(torch.zeros(1))
    def forward(self,logits): return logits/(torch.exp(self.logT)+1e-6)

def fit_temperature(model,dl,max_iter=200,lr=0.05):
    ts=TempScale(); opt=torch.optim.LBFGS(ts.parameters(),lr=lr,max_iter=max_iter,line_search_fn='strong_wolfe'); nll=nn.CrossEntropyLoss()
    logits_list=[]; y_list=[]
    with torch.no_grad():
        for xb,yb in dl:
            logits_list.append(model(xb)); y_list.append(yb)
    logits=torch.cat(logits_list,0); y=torch.cat(y_list,0)
    def closure(): opt.zero_grad(); loss=nll(ts(logits),y); loss.backward(); return loss
    opt.step(closure); T=float(torch.exp(ts.logT).item()); return ts,T

temp_module,T_value=fit_temperature(model,val_dl)
json.dump({'T':T_value},open('calibration.json','w')); print('Temperature:',T_value)

In [ ]:
def evaluate_with_gate(model,temp_module,dl,tau=0.5,costs=None):
    if costs is None: costs={'FN':5.0,'FP':1.0}
    import numpy as np
    tot=0.0; yT=[]; yP=[]
    with torch.no_grad():
        for xb,yb in dl:
            logits=temp_module(model(xb)); probs=torch.softmax(logits,dim=-1).cpu().numpy(); pred=probs.argmax(axis=1)
            low=(probs.max(axis=1)<tau); pred[low]=0; yT.append(yb.numpy()); yP.append(pred)
            tot-=costs['FN']*np.sum((pred!=yb.numpy()) & (yb.numpy()!=0)); tot-=costs['FP']*np.sum((pred!=yb.numpy()) & (yb.numpy()==0))
    y_true=np.concatenate(yT); y_pred=np.concatenate(yP); return tot/len(y_true), y_true, y_pred

import numpy as np
best_tau=None; best_util=-1e9; best_pair=None
for t in np.linspace(0.3,0.9,13):
    u,yv,yp=evaluate_with_gate(model,temp_module,val_dl,tau=float(t))
    if u>best_util: best_util=u; best_tau=float(t); best_pair=(yv,yp)
json.dump({'best_tau':best_tau,'utility':best_util},open('gate_tau.json','w'))
y_true_val,y_pred_val=best_pair
print('Best tau:',best_tau,'utility:',best_util)

In [ ]:
from sklearn.metrics import confusion_matrix

def _plot_cm(cm,labels,title,fname):
    import matplotlib.pyplot as plt
    fig=plt.figure(figsize=(6,5)); im=plt.imshow(cm,interpolation='nearest'); plt.title(title); plt.colorbar(im,fraction=0.046,pad=0.04)
    ticks=np.arange(len(labels)); plt.xticks(ticks,labels,rotation=45,ha='right'); plt.yticks(ticks,labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j,i,f"{cm[i,j]:d}",ha='center',va='center',color=('white' if cm[i,j]>cm.max()/2 else 'black'),fontsize=8)
    plt.ylabel('True'); plt.xlabel('Pred'); plt.tight_layout(); plt.savefig(fname,dpi=160); plt.close()

cm_overall=confusion_matrix(y_true_val,y_pred_val,labels=list(range(len(ACTIONS))))
_plot_cm(cm_overall,ACTIONS,'Confusion matrix - overall (val)','cm_overall_v5.png')
val_mask=(df['day_index']==df['day_index'].max()).values
val_df=df[df['day_index']==df['day_index'].max()].copy(); val_df['hour']=pd.to_datetime(val_df['timestamp']).dt.hour
hour_counts=val_df.groupby('hour')['hour'].transform('count'); val_df['load_quartile']=pd.qcut(hour_counts,q=4,labels=False,duplicates='drop')
loadq_val=val_df['load_quartile'].values; daytypes_val=val_df['daytype'].values
for key in np.unique(daytypes_val):
    m=(daytypes_val==key); cm=confusion_matrix(y_true_val[m],y_pred_val[m],labels=list(range(len(ACTIONS))))
_plot_cm(cm,ACTIONS,f'CM by daytype={key}',f'cm_daytype_{key}.png')
valid_mask=~pd.isna(loadq_val)
for key in np.unique(loadq_val[valid_mask]):
    m=(loadq_val==key); cm=confusion_matrix(y_true_val[m],y_pred_val[m],labels=list(range(len(ACTIONS))))
_plot_cm(cm,ACTIONS,f'CM by loadquartile={int(key)}',f'cm_loadq_{int(key)}.png')
print('Saved CM images.')

In [ ]:
import json, numpy as np
pred_dump={'y_true':np.array(y_true_val).tolist(),'y_pred':np.array(y_pred_val).tolist(),'actions':ACTIONS}
json.dump(pred_dump,open('gru_preds.json','w'))
print('Saved gru_preds.json')

In [ ]:
import base64, json, os
from sklearn.metrics import classification_report, accuracy_score
from string import Template
import pandas as pd

def _img_data_uri(path):
    with open(path,'rb') as f: return 'data:image/png;base64,'+base64.b64encode(f.read()).decode('ascii')
acc=accuracy_score(y_true_val,y_pred_val)
report=classification_report(y_true_val, y_pred_val, labels=list(range(len(ACTIONS))), target_names=ACTIONS, output_dict=True, zero_division=0)
report_df=pd.DataFrame(report).T
try: tau=json.load(open('gate_tau.json'))['best_tau']
except Exception: tau='-'
try: T=json.load(open('calibration.json'))['T']
except Exception: T='-'
overall_uri=_img_data_uri('cm_overall_v5.png')
rows_daytype=''
for p in sorted([p for p in os.listdir('.') if p.startswith('cm_daytype_') and p.endswith('.png')]):
    key=p.replace('cm_daytype_','').replace('.png',''); rows_daytype+="<tr><td class='left'>"+key+"</td><td><img src='"+_img_data_uri(p)+"' width='360'/></td></tr>"
rows_loadq=''
for p in sorted([p for p in os.listdir('.') if p.startswith('cm_loadq_') and p.endswith('.png')]):
    key=p.replace('cm_loadq_','').replace('.png',''); rows_loadq+="<tr><td class='left'>"+key+"</td><td><img src='"+_img_data_uri(p)+"' width='360'/></td></tr>"
report_html=report_df.to_html(index=True,classes='tbl',float_format=lambda x: f"{x:.3f}")

tpl=Template("""<!doctype html><html lang='de'><head><meta charset='utf-8'><title>ED PoC - Clinician Brief v5</title><style>body { font-family: -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif; margin:24px;} h1,h2 { margin:0.2em 0;} .card { border:1px solid #ddd; border-radius:12px; padding:16px; margin:12px 0;} table { border-collapse: collapse; width:100%; font-size:14px;} th,td { border:1px solid #ddd; padding:6px 8px; text-align:right;} th { background:#f7f7f7;} .left { text-align:left;} .small { color:#666; font-size:12px;} </style></head><body><h1>ED Coordination Assistant - Brief v5 (DE)</h1><div class='small'>Synthetic demo - Kein Zugriff auf Echtdaten - Europe/Berlin</div><div class='card'><h2>Zweck</h2><p>Uebersicht der Aktionsqualitaet des GRU-Headers (Baseline+Stress). <b>Kein Diagnosesystem.</b></p><ul><li><b>Aktionsraum:</b> $actions</li><li><b>n</b> = $n_samples Validierungsbeispiele</li><li><b>Accuracy</b>: $acc</li><li><b>Gate tau</b>: $tau &nbsp; · &nbsp; <b>Temperatur</b>: $T</li></ul></div><div class='card'><h2>Konfusionsmatrix (Overall)</h2><img alt='Konfusionsmatrix' src='$overall_uri' width='480'/></div><div class='card'><h2>Stratifizierte Matrizen</h2><h3>Nach Tagtyp</h3><table><tr><th class='left'>Tagtyp</th><th>Matrix</th></tr>$rows_daytype</table><h3>Nach Lastquartil</h3><table><tr><th class='left'>Quartil</th><th>Matrix</th></tr>$rows_loadq</table></div><div class='card'><h2>Klassenmetriken</h2>$report_html</div><div class='card'><h2>Guardrails (Erinnerung)</h2><ul><li><b>Keine</b> Diagnose-/Therapie-Empfehlungen.</li><li><b>Human-in-the-loop</b> fuer jede Order/Anmeldung.</li><li><b>Kapazitaet</b> mit Stale-Flag - Re-Polling/Verifikation.</li><li><b>Audit-Log</b> fuer Vorschlaege, Overrides, Eskalationen.</li></ul></div></body></html>""")
html=tpl.substitute(actions=', '.join(ACTIONS),n_samples=str(len(y_true_val)),acc=f"{acc:.3f}",tau=str(tau),T=str(T),overall_uri=overall_uri,rows_daytype=rows_daytype,rows_loadq=rows_loadq,report_html=report_html)
open('ED_PoC_Clinician_Brief_DE_v5.html','w',encoding='utf-8').write(html)
import zipfile
with zipfile.ZipFile('ED_PoC_Clinician_Brief_DE_v5.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write('ED_PoC_Clinician_Brief_DE_v5.html'); z.write('cm_overall_v5.png')
    [z.write(p) for p in os.listdir('.') if (p.startswith('cm_daytype_') and p.endswith('.png')) or (p.startswith('cm_loadq_') and p.endswith('.png'))]
    for opt in ['gate_tau.json','calibration.json','gru_preds.json']:
        if os.path.exists(opt): z.write(opt)
print('Rendered ED_PoC_Clinician_Brief_DE_v5.html and zipped bundle.')